# Second-phase sentiment-head retrain — ablation launcher (thin)

**All logic lives in `.py` scripts; this notebook only calls them** (Colab is costly — keep compute in tested scripts, not notebook cells).

- Train one arm: `scripts/training/train_sentiment_arm.py --arm <ARM>`
- Eval on test split: `evaluate_e2e_pipeline.py` → `analyze_eval_gaps.py` → `calibration_baseline.py`
- Compare arms (run locally after all arms): `scripts/evaluation/compare_arms.py`

Set `ARM` in cell 2, run top-to-bottom, then re-run with another `ARM`.

| ARM | encoder | loss | purpose |
|---|---|---|---|
| `armA` | frozen | legacy MSE+(1−Pearson) | data effect (control) |
| `control` | frozen | CCC + weighted-Huber | recipe effect, head-only |
| `armC` | **top-2 layers unfrozen** | CCC + weighted-Huber | discrimination lever (expected winner) |
| `armD` | frozen | weighted-Huber only | isolate magnitude weighting |
| `armE` | frozen | CCC + Huber, no sign | isolate sign penalty |

All arms start from v2.0, select best by **CCC**, eval on the held-out **test** split.

In [ ]:
# 1. Mount + GPU
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Config: pick ARM + RUN_TAG (training/eval logic is in scripts/)
import os
ARM = 'armC'      # armA | control | armC | armD | armE
# RUN_TAG must be STABLE across re-runs so an interrupted run can auto-resume:
# re-running the SAME ARM+RUN_TAG continues from last.pt (no wasted epochs).
# Change RUN_TAG (e.g. 'v2', 'enc_lr5e7') to start a FRESH independent run.
RUN_TAG = 'v1'
PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
os.chdir(PROJECT_PATH)
RUN_ID    = f"{ARM}__{RUN_TAG}"
TEST_FILE = f"{PROJECT_PATH}/data/labeled/deepseek_t1/splits/test.jsonl"
CKPT_DIR  = f"{PROJECT_PATH}/checkpoints/retrain_{RUN_ID}"   # stable per run -> resume + no cross-run overwrite
OUT       = f"{PROJECT_PATH}/outputs/retrain_{RUN_ID}"
assert os.path.exists(PROJECT_PATH), PROJECT_PATH
assert os.path.exists(f"{PROJECT_PATH}/scripts/training/train_sentiment_arm.py")
print('RUN_ID:', RUN_ID, '| resumes from last.pt if present')
print('ckpt  :', CKPT_DIR)
print('out   :', OUT)

In [ ]:
# 3. Deps
!pip install -q transformers torch torchvision torchaudio pytorch-crf

In [ ]:
# 4. Train this arm — all logic in scripts/training/train_sentiment_arm.py
#    Writes the checkpoint to CKPT_DIR (per-run) and APPENDS a row to
#    outputs/retrain_runs.jsonl (append-only ledger of every run — never overwritten).
#    Override hyperparams via flags, e.g. --batch-size 32 if OOM, --head-lr 3e-4.
!cd "{PROJECT_PATH}" && python scripts/training/train_sentiment_arm.py \
    --arm {ARM} --tag {RUN_TAG} --ckpt-dir "{CKPT_DIR}" --project "{PROJECT_PATH}"

In [ ]:
# 5. Eval the retrained run on the held-out TEST split (all via scripts):
#    e2e pipeline -> gap analysis -> calibration floor. Outputs -> OUT (per-run, no overwrite).
BEST = f"{CKPT_DIR}/best_model.pt"
os.makedirs(OUT, exist_ok=True)
LOCAL_EVAL = f"/content/eval_{RUN_ID}"
!cd "{PROJECT_PATH}" && python scripts/evaluation/evaluate_e2e_pipeline.py \
    --checkpoint "{BEST}" --benchmark "{TEST_FILE}" \
    --output-dir "{OUT}" --local-output-dir "{LOCAL_EVAL}" \
    --ner-mode single-pass --inference-batch-size 16 --iou-threshold 0.5 --save-predictions
import glob
preds = sorted(glob.glob(f'{LOCAL_EVAL}/e2e_predictions_*.jsonl') + glob.glob(f'{OUT}/e2e_predictions_*.jsonl'))[-1]
!cd "{PROJECT_PATH}" && python scripts/evaluation/analyze_eval_gaps.py \
    --predictions "{preds}" --final "{TEST_FILE}" \
    --person-scores "{PROJECT_PATH}/data/labeled/deepseek_t1/person_scores.jsonl" --output-dir "{OUT}"
!cd "{PROJECT_PATH}" && python scripts/evaluation/calibration_baseline.py --predictions "{preds}" --output "{OUT}/calibration_baseline.json"
print('\n=== gap analysis (RUN_ID=' + RUN_ID + ') ===')
print(open(f'{OUT}/gap_analysis.md').read())

In [ ]:
# 6. CROSS-CHECK any checkpoint on the Sonnet holdout (GOLD-grade NER + sentiment).
#    Decisive promote/no-promote test: does touching the encoder degrade NER, and does
#    the sentiment gain hold cross-distribution? STANDALONE — only needs cells 1 (mount)
#    + 3 (deps). ~30 min for 6,750 articles. Edit CKPT_TO_CHECK + LABEL below.
import os, glob
PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
os.chdir(PROJECT_PATH)

# ---- pick the model to cross-check (uncomment one) ----
LABEL = "control";    CKPT_TO_CHECK = f"{PROJECT_PATH}/checkpoints/retrain_control/best_model.pt"
# LABEL = "armC__v1";   CKPT_TO_CHECK = f"{PROJECT_PATH}/checkpoints/retrain_armC__v1/best_model.pt"

SONNET = f"{PROJECT_PATH}/data/labeled/final/holdout_relabeled.jsonl"
OUT_SONNET = f"{PROJECT_PATH}/outputs/retrain_{LABEL}_sonnet"
os.makedirs(OUT_SONNET, exist_ok=True)
assert os.path.exists(CKPT_TO_CHECK), f"checkpoint not found: {CKPT_TO_CHECK}"
assert os.path.exists(SONNET), SONNET
!cd "{PROJECT_PATH}" && python scripts/evaluation/evaluate_e2e_pipeline.py \
    --checkpoint "{CKPT_TO_CHECK}" --benchmark "{SONNET}" \
    --output-dir "{OUT_SONNET}" --local-output-dir "/content/sonnet_{LABEL}" \
    --ner-mode single-pass --inference-batch-size 16 --iou-threshold 0.5
# Compare vs v2.0's stored numbers on the SAME Sonnet holdout (NER + sentiment).
m = sorted(glob.glob(f'{OUT_SONNET}/e2e_metrics_*.json'))[-1]
!cd "{PROJECT_PATH}" && python scripts/evaluation/compare_on_sonnet.py \
    --arm "{m}" --arm-label "{LABEL}" --output "{OUT_SONNET}/sonnet_crosscheck.md"
print(open(f'{OUT_SONNET}/sonnet_crosscheck.md').read())

In [ ]:
# 10. (Run when all arms done) Terminate runtime
from google.colab import runtime
runtime.unassign()